# **Data Preparation Notebook **

## Objectives

* Prepare data for model training
  * Impute missing values
  * Ensure categorical variables are encoded
  * Consider dropping outliers 

## Inputs

outputs/datasets/collection/house_prices

## Outputs

outputs/datasets/cleaned/house_prices


---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [ ]:
import os
current_dir = os.getcwd()
current_dir

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [ ]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

Confirm the new current directory

In [ ]:
current_dir = os.getcwd()
current_dir

# Data Preparation



In [ ]:
import pandas as pd
df = pd.read_csv(f"outputs/datasets/collection/house_prices")
df.head()

---

# Missing Values

In [ ]:
# Display a table to visualise null data
null_data = []

for col in df.columns:
    null_count = df[col].isnull().sum()
    if null_count > 0:
        null_data.append({
            'Variable': col,
            'Null Count': null_count,
            'Null %': round(null_count / len(df) * 100, 1),
            'Data Type': 'Numerical' if df[col].dtype in ['int64', 'float64'] else 'Categorical'
        })

null_df = pd.DataFrame(null_data).sort_values('Null %', ascending=False)
print(null_df.to_string(index=False))

We will be imputing missing data for the following reasons:
* To avoid loss of statistical power, compared to dropping rows with null values
* To reduce bias
* Ensure compatability with algorithms

We have chosen to impute median values for numerical values. The reasoning behind this is housing attributes data is likely to be skewed, with a small number of large properties affecting the mode. For this reason, we will be using median.

With categorical values, we will be imputing the mode value.



In [ ]:
from sklearn.impute import SimpleImputer

# Median imputation for numerical variables
median_imputer = SimpleImputer(strategy='median')
median_cols = ['LotFrontage', 'GarageYrBlt', 'BedroomAbvGr', '2ndFlrSF', 'MasVnrArea']
df[median_cols] = median_imputer.fit_transform(df[median_cols])

# Mode imputation for categorical variables
mode_imputer = SimpleImputer(strategy='most_frequent')
mode_cols = ['BsmtExposure', 'BsmtFinType1', 'GarageFinish']
df[mode_cols] = mode_imputer.fit_transform(df[mode_cols])

# Confirm no null values remain
print(df.isnull().sum())

---

## Encode Categorical Values

As in Notebook 2, we will encode categorical values to ensure compliance with algorithms.

In [ ]:
from feature_engine.encoding import OrdinalEncoder


encoder = OrdinalEncoder(
    encoding_method='arbitrary',
    variables=['KitchenQual', 'BsmtExposure', 'BsmtFinType1', 'GarageFinish'],
    missing_values='ignore'
)


df_encoded = encoder.fit_transform(df)
print(df_encoded.shape)
df_encoded.head(5)

---

## Handling outliers

It is worth considering whether to drop or retain outliers when preparing data for model training. 

First we will explore and visualise outlying data using a histogram, and sceondly calculate the number of percentage and outliers.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# This code was taken and modified from Feature Engine Unit 6 of Code Institute's Full Stack program.
def plot_histogram_and_boxplot(df, col):
    fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(7, 7), 
                             gridspec_kw={"height_ratios": (.15, .85)})
    sns.boxplot(data=df, x=col, ax=axes[0])
    sns.histplot(data=df, x=col, kde=True, ax=axes[1])
    fig.suptitle(f"{col} Distribution - Boxplot and Histogram")
    plt.show()
    
    IQR = df[col].quantile(q=0.75) - df[col].quantile(q=0.25)
    print(
        f"This is the range where a datapoint is not an outlier: from "
        f"{(df[col].quantile(q=0.25) - 1.5*IQR).round(2)} to "
        f"{(df[col].quantile(q=0.75) + 1.5*IQR).round(2)}"
    )
    print("\n")

plot_histogram_and_boxplot(df, 'SalePrice')

In [ ]:
lower_bound = 3937.5
upper_bound = 340037.5

outliers = df[(df['SalePrice'] < lower_bound) | (df['SalePrice'] > upper_bound)]

print(f"Number of outliers: {len(outliers)}")
print(f"Percentage of dataset: {round(len(outliers) / len(df) * 100, 1)}%")

## Observations and Conclusion on handling outliers

We consider the following:
* The range where a datapoint is not an outlier: from 3937.5 to 340037.5
* Number of outliers: 61
* Percentage of dataset: 4.2%

4.2% is a small enough percentage to consider dropping, however we have decided to include outliers initially for the following reasons:
* Real life houses vary as reflected in the dataset, which we want to account for in our model.
* We are expecting to use a Random Forest model, which handles outliers well.
* Dropping data would restrict our dataset.

We will return to this decision if we encounter issues with our model training.




# Push files to Repo

We will now save our cleaned data set 

In [ ]:
# Check dataframe before saving
df_encoded.info()

In [ ]:
# Create collections folder and save cleaned dataset
import os
try:
    os.makedirs(name='outputs/datasets/cleaned')
except Exception as e:
    print(e)

df_encoded.to_csv('outputs/datasets/cleaned/house_prices', index=False)

## Split Test and Train Sets

We will now split our dataset in preparation for Feature Engineering and Modelling.

In [ ]:
from sklearn.model_selection import train_test_split
TrainSet, TestSet, _, __ = train_test_split(
                                        df_encoded,
                                        df_encoded['SalePrice'],
                                        test_size=0.2,
                                        random_state=27)

print(f"TrainSet shape: {TrainSet.shape} \nTestSet shape: {TestSet.shape}")

In [ ]:
TrainSet.to_csv("outputs/datasets/cleaned/TrainSetCleaned.csv", index=False)

In [ ]:
TestSet.to_csv("outputs/datasets/cleaned/TestSetCleaned.csv", index=False)